In [ ]:
import numpy as np
import pandas as pd
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X = pd.read_csv('train.csv')
cat_features = ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 'Exercise angina', 'Slope of ST',
                'Number of vessels fluro', 'Thallium', 'Age_bin', 'BP_bin', 'Cholesterol_bin', 'MaxHR_bin',
                'ST_bin']

#Bin
X['Age_bin'] = pd.cut(X['Age'], bins=5).astype(str)
X['BP_bin'] = pd.cut(X['BP'], bins=5).astype(str)
X['Cholesterol_bin'] = pd.cut(X['Cholesterol'], bins=5).astype(str)
X['MaxHR_bin'] = pd.cut(X['Max HR'], bins=5).astype(str)
X['ST_bin'] = pd.cut(X['ST depression'], bins=5).astype(str)

#Digit
X['Age_units'] = X['Age'] % 10
X['Age_tens'] = (X['Age'] // 10) % 10

X['BP_units'] = X['BP'] % 10
X['BP_tens'] = (X['BP'] // 10) % 10
X['BP_hundreds'] = (X['BP'] // 100) % 10

X['Chol_units'] = X['Cholesterol'] % 10
X['Chol_tens'] = (X['Cholesterol'] // 10) % 10
X['Chol_hundreds'] = (X['Cholesterol'] // 100) % 10

X['MaxHR_units'] = X['Max HR'] % 10
X['MaxHR_tens'] = (X['Max HR'] // 10) % 10
X['MaxHR_hundreds'] = (X['Max HR'] // 100) % 10

X['St_depression_1'] = (X['ST depression'] * 10 % 10).astype(int)

X.replace({'Heart Disease': {'Presence': 1, 'Absence': 0}}, inplace=True)
y = X['Heart Disease'].astype(int)
X.drop(['Heart Disease', 'id'], axis=1, inplace=True)


def objective(trial: optuna.Trial) -> float:
    params = {
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),

        'iterations': 1000,
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'random_seed': 42,
        'task_type': 'GPU',
        'devices': '0',
        'verbose': False
    }

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_index, test_index in kf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        train_pool = Pool(X_train, y_train, cat_features=cat_features)
        test_pool = Pool(X_test, y_test, cat_features=cat_features)

        model = CatBoostClassifier(**params)

        model.fit(train_pool,
                  eval_set=test_pool,
                  early_stopping_rounds=100)

        preds = model.predict_proba(test_pool)[:, 1]
        score = roc_auc_score(y_test, preds)
        scores.append(score)

    return np.mean(scores)


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=500)

print('Number of finished trials: {}'.format(len(study.trials)))

print('Best trial:')
trial = study.best_trial

print('Value: {}'.format(trial.value))

print('Params: ')
for key, value in trial.params.items():
    print('{}: {}'.format(key, value))

[I 2026-05-06 23:59:09,386] A new study created in memory with name: no-name-be6dd35c-b55d-4b50-b18a-9445222b28fa
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-05-07 00:02:34,326] Trial 0 finished with value: 0.9551754797464156 and parameters: {'iterations': 1627, 'depth': 3, 'learning_rate': 0.02051987277177882, 'l2_leaf_reg': 1.202373387720176}. Best is trial 0 with value: 0.9551754797464156.
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default me

In [21]:
X = pd.read_csv('test.csv')

#Bin
X['Age_bin'] = pd.cut(X['Age'], bins=5).astype(str)
X['BP_bin'] = pd.cut(X['BP'], bins=5).astype(str)
X['Cholesterol_bin'] = pd.cut(X['Cholesterol'], bins=5).astype(str)
X['MaxHR_bin'] = pd.cut(X['Max HR'], bins=5).astype(str)
X['ST_bin'] = pd.cut(X['ST depression'], bins=5).astype(str)

0.955525302701582 and parameters: {'iterations': 1209, 'depth': 5, 'learning_rate': 0.08375766130052575, 'l2_leaf_reg': 3.9695263497534183

#Digit
X['Age_units'] = X['Age'] % 10
X['Age_tens'] = (X['Age'] // 10) % 10

X['BP_units'] = X['BP'] % 10
X['BP_tens'] = (X['BP'] // 10) % 10
X['BP_hundreds'] = (X['BP'] // 100) % 10

X['Chol_units'] = X['Cholesterol'] % 10
X['Chol_tens'] = (X['Cholesterol'] // 10) % 10
X['Chol_hundreds'] = (X['Cholesterol'] // 100) % 10

X['MaxHR_units'] = X['Max HR'] % 10
X['MaxHR_tens'] = (X['Max HR'] // 10) % 10
X['MaxHR_hundreds'] = (X['Max HR'] // 100) % 10

id = X['id']
X.drop('id', axis=1, inplace=True)
preds = model.predict_proba(X)[:, 1]

sample_submission = pd.DataFrame(data={'id': id, 'Heart Disease': preds})
sample_submission.to_csv('sample_submission.csv', index=False)